In [ ]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import re
import string
import warnings
from collections import Counter
import pickle

warnings.filterwarnings('ignore')

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

print("Libraries imported successfully!")

In [ ]:
df = pd.read_csv('clinical_trial_dataset.csv')

print("Dataset Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nFirst 5 rows:")
df.head()

In [ ]:
print("Dataset Info:")
df.info()
print("\nStatistical Summary:")
df.describe()
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
df = df.dropna(subset=['Brief Summary'])
df = df.drop_duplicates()
print(f"Dataset after cleaning: {df.shape}")

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['Cleaned_Summary'] = df['Brief Summary'].apply(clean_text)

print("Sample cleaned text:")
print(df['Cleaned_Summary'].iloc[0][:500])

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

df['Processed_Summary'] = df['Cleaned_Summary'].apply(preprocess_text)

print("Sample processed text:")
print(df['Processed_Summary'].iloc[0][:500])

In [ ]:
if 'Disease_Category' in df.columns:
    plt.figure(figsize=(12, 6))
    df['Disease_Category'].value_counts().plot(kind='bar')
    plt.title('Disease Category Distribution')
    plt.xlabel('Disease Category')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    print("Category Distribution:")
    print(df['Disease_Category'].value_counts())

In [ ]:
all_words = ' '.join(df['Processed_Summary']).split()
word_freq = Counter(all_words)
common_words = word_freq.most_common(20)

words, counts = zip(*common_words)

plt.figure(figsize=(12, 6))
plt.bar(words, counts)
plt.title('Top 20 Most Common Words')
plt.xlabel('Words')
plt.ylabel('Frequency')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
from wordcloud import WordCloud

wordcloud = WordCloud(width=800, height=400, background_color='white').generate(' '.join(df['Processed_Summary']))

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Clinical Trial Summaries')
plt.show()

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X = tfidf.fit_transform(df['Processed_Summary']).toarray()

le = LabelEncoder()
y = le.fit_transform(df['Disease_Category'])

print("TF-IDF Shape:", X.shape)
print("Unique Classes:", len(le.classes_))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print("Logistic Regression Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_lr, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_lr, average='weighted'):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_lr, average='weighted'):.4f}")

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_rf, average='weighted'):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_rf, average='weighted'):.4f}")

In [ ]:
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

print("Multinomial NB Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_nb):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_nb, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_nb, average='weighted'):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_nb, average='weighted'):.4f}")

In [ ]:
svm = SVC(kernel='linear', random_state=42)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

print("SVM Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_svm, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_svm, average='weighted'):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_svm, average='weighted'):.4f}")

In [ ]:
models = ['Logistic Regression', 'Random Forest', 'Naive Bayes', 'SVM']
accuracies = [
    accuracy_score(y_test, y_pred_lr),
    accuracy_score(y_test, y_pred_rf),
    accuracy_score(y_test, y_pred_nb),
    accuracy_score(y_test, y_pred_svm)
]

plt.figure(figsize=(10, 6))
bars = plt.bar(models, accuracies, color=['blue', 'green', 'red', 'purple'])
plt.ylim(0, 1)
plt.title('Model Accuracy Comparison')
plt.ylabel('Accuracy')
plt.xlabel('Models')

for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{acc:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

In [ ]:
cm = confusion_matrix(y_test, y_pred_lr)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Logistic Regression')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
print("Classification Report - Logistic Regression:")
print(classification_report(y_test, y_pred_lr, target_names=le.classes_))

In [ ]:
cv_scores = cross_val_score(lr, X, y, cv=5)
print(f"Cross-Validation Scores: {cv_scores}")
print(f"Mean CV Score: {cv_scores.mean():.4f}")
print(f"Standard Deviation: {cv_scores.std():.4f}")

In [ ]:
# Save the best model (Logistic Regression)
with open('disease_classification_model.pkl', 'wb') as f:
    pickle.dump(lr, f)

# Save TF-IDF Vectorizer
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

# Save Label Encoder
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print("Models saved successfully!")

In [ ]:
def predict_disease(text):
    cleaned = clean_text(text)
    processed = preprocess_text(cleaned)
    vectorized = tfidf.transform([processed])
    prediction = lr.predict(vectorized)
    disease = le.inverse_transform(prediction)[0]
    probs = lr.predict_proba(vectorized)[0]
    confidence = max(probs)
    return disease, confidence

# Test prediction
test_summary = "This study evaluates the efficacy of a new drug for treating breast cancer patients."
disease, confidence = predict_disease(test_summary)
print(f"Predicted Disease: {disease}")
print(f"Confidence: {confidence:.4f}")

In [ ]:
if 'Disease_Category' in df.columns:
    fig = px.bar(df['Disease_Category'].value_counts().reset_index(), 
                 x='index', y='Disease_Category',
                 title='Disease Category Distribution (Plotly)',
                 labels={'index': 'Disease Category', 'Disease_Category': 'Count'})
    fig.show()

In [ ]:
model_results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'Naive Bayes', 'SVM'],
    'Accuracy': accuracies
})

fig = px.bar(model_results, x='Model', y='Accuracy', 
             title='Model Performance Comparison',
             color='Model',
             text='Accuracy')
fig.update_traces(texttemplate='%{text:.4f}', textposition='outside')
fig.update_layout(yaxis_range=[0, 1])
fig.show()

In [ ]:
df.to_csv('processed_clinical_trial_data.csv', index=False)
print("Processed data saved successfully!")

In [ ]:
print("="*60)
print("PROJECT SUMMARY")
print("="*60)
print(f"Total Samples: {len(df)}")
print(f"Number of Disease Categories: {len(le.classes_)}")
print(f"Best Model: Logistic Regression")
print(f"Best Accuracy: {max(accuracies):.4f}")
print("="*60)
print("Project Completed Successfully!")